# Laubmann-KG — full workflow (all 34 volumes, Colab)

One notebook, three stages, all reading/writing your Drive:

| stage | what | output |
|---|---|---|
| **A** | build the text + multimodal corpus from `Laubmann_NN_gemini/` region JSONs | `corpus_<date>/` |
| **B** | detect duplicate pages → human review → apply decisions | `corpus_<date>_dedup/` |
| **C** | LLM extraction (Gemini) → QA → GBIF/Wikidata linking → RDF/JSON-LD + SHACL + DwC-A for **all 34 volumes** | `kg_exports_<tag>/` |

Stage B has a **human step in the middle**: download `review.html`, adjudicate the
clusters, export `dedup_decisions.json`, upload it back, then continue.
Stage C has an optional **smoke test** (25 entries live) before the full run, and
produces **review CSVs** (taxon/person links, QA flags) for adjudication.
Each stage is skippable if its output already exists — re-running the notebook
after a disconnect is safe (LLM calls, GBIF/Wikidata lookups are cached on Drive).

## 0 · Mount Drive, clone repo, install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO = '/content/laubmann-kg_TP'
if not os.path.isdir(REPO):
    !git clone https://github.com/Maelkolb/laubmann-kg_TP.git {REPO}
else:
    # the yaml-rewrite cell below modifies configs/full_llm.yaml; restore it before pulling
    !cd {REPO} && git checkout -- configs/full_llm.yaml && git pull --ff-only
%cd {REPO}
!git log --oneline -1

!pip -q install -e ".[llm]"
!pip -q install -U google-genai            # thinking_level support needs a recent SDK
!pip -q install rapidfuzz datasketch imagehash pillow pandas

ADDONS = f'{REPO}/HistOrniGraph_addons'
DEDUP  = f'{ADDONS}/dedup'
assert os.path.isdir(f'{ADDONS}/laubmann_corpus'), 'addons missing — repo out of date?'
assert os.path.isdir(f'{REPO}/src/laubmann_kg/linking'), 'linking stage missing — GitHub main is behind (push it first)'
print('repo + addons ready')

## 1 · Config — edit these

In [ ]:
from pathlib import Path

# Root on Drive holding the Laubmann_NN_gemini/ folders (each with regions/*.json):
OUTPUT_BASE  = Path('/content/drive/MyDrive/HistOrniGraph_output')

# The corpus this run works on (stage A writes it, stages B/C read it):
CORPUS_DIR   = OUTPUT_BASE / 'corpus_2026-07-21'
CORPUS_DEDUP = OUTPUT_BASE / (CORPUS_DIR.name + '_dedup')

# One tag per pipeline version -> a FRESH export folder (never mix old and new files).
RUN_TAG      = '2026-08-18'      # 0.4.0 export (kg_exports_2026-08-18 on Drive)
EXPORTS_DIR  = OUTPUT_BASE / f'kg_exports_{RUN_TAG}'      # rdf/ jsonld/ dwca/ review/
SMOKE_DIR    = OUTPUT_BASE / f'kg_smoke_{RUN_TAG}'        # smoke-test output (small)

# Caches on Drive: interrupted runs resume for free across sessions.
LLM_CACHE    = OUTPUT_BASE / 'llm_cache_v2'     # extraction responses (v2 = new prompt; old llm_cache is dead)
LINK_CACHE   = OUTPUT_BASE / 'linking_cache'    # gbif_cache.json, wikidata_cache.json, llm/ (folk-name proposer)

# Diary volumes only — vol 35 is the general index/register, keep it excluded.
VOLUMES = list(range(1, 35))

for k, v in [('OUTPUT_BASE', OUTPUT_BASE), ('CORPUS_DIR', CORPUS_DIR),
             ('CORPUS_DEDUP', CORPUS_DEDUP), ('EXPORTS_DIR', EXPORTS_DIR),
             ('LLM_CACHE', LLM_CACHE), ('LINK_CACHE', LINK_CACHE)]:
    print(f'{k:13s}', v, '✓ exists' if v.exists() else '(will be created)')

In [ ]:
# Gemini key from Colab secrets (Runtime ▸ Secrets ▸ add LST_Gemini)
import os
from google.colab import userdata
key = userdata.get('LST_Gemini')
os.environ['GOOGLE_API_KEY'] = key
os.environ['GEMINI_API_KEY'] = key
print('Gemini key loaded:', bool(key))

## Stage A · Build the corpus (skip if `CORPUS_DIR` already exists)

Standard-library only; ~34 volumes take a few minutes of Drive I/O.

In [ ]:
if (CORPUS_DIR / 'corpus.json').exists():
    print('corpus already built at', CORPUS_DIR, '— skipping stage A')
else:
    vols = ' '.join(map(str, VOLUMES))
    !cd "{ADDONS}" && python build_corpus.py \
        --output-base "{OUTPUT_BASE}" \
        --corpus-dir  "{CORPUS_DIR}" \
        --per-volume --multimodal --report --volumes {vols}

In [ ]:
# Sanity: size + the known Vol-1→Vol-15 cross-run contamination check
import json
from collections import defaultdict
pages = json.load(open(CORPUS_DIR / 'corpus.json'))
print(len(pages), 'pages')

by_pid = defaultdict(set)
for p in pages:
    by_pid[p['page_id']].add(int(p['volume']))
cross = {pid: sorted(v) for pid, v in by_pid.items() if len(v) > 1}
print('page_ids appearing in >1 volume:', len(cross),
      '→ dedup will cluster these' if cross else '→ no cross-volume contamination')

## Stage B1 · Detect duplicates + build the review GUI

In [ ]:
DEDUP_OUT = CORPUS_DIR / 'dedup'

!cd "{DEDUP}" && python detect_duplicates.py "{CORPUS_DIR}/corpus.json" \
    -o "{DEDUP_OUT}" \
    --scan-window 3 --cluster-threshold 0.55 --high-threshold 0.80 \
    --image-root "{OUTPUT_BASE}"

!cd "{DEDUP}" && python build_review_gui.py "{DEDUP_OUT}/duplicates_report.jsonl" \
    --corpus "{CORPUS_DIR}/corpus.json" \
    -o "{DEDUP_OUT}/review.html"

import pandas as pd
dup = pd.read_csv(DEDUP_OUT / 'duplicates_report.csv')
print(len(dup), 'clusters flagged —', (dup.suggested_action == 'drop_duplicates').sum(), 'auto-drop (≥0.80),',
      (dup.suggested_action != 'drop_duplicates').sum(), 'need review')
dup.head(10)

In [ ]:
# Download review.html → adjudicate → the GUI exports dedup_decisions.json.
# (It is also on Drive at DEDUP_OUT/review.html if the download times out.)
from google.colab import files
files.download(str(DEDUP_OUT / 'review.html'))

## Stage B2 · Apply decisions → clean corpus

Put the exported `dedup_decisions.json` into `CORPUS_DIR/dedup/` on Drive
(or upload it here). Non-destructive: writes a NEW `corpus_*_dedup/` with
regenerated `entries.csv` (incl. `entry_uid`/`page_uid`/`region_uid`) and a
`dedup_manifest` recording every dropped page.

In [ ]:
DEDUP_OUT = CORPUS_DIR / 'dedup'
DECISIONS = DEDUP_OUT / 'dedup_decisions.json'
if not DECISIONS.exists():
    from google.colab import files
    up = files.upload()                      # pick dedup_decisions.json
    (DEDUP_OUT).mkdir(parents=True, exist_ok=True)
    Path(DECISIONS).write_bytes(next(iter(up.values())))

!python "{DEDUP}/apply_dedup.py" \
    --corpus-dir "{CORPUS_DIR}" \
    --decisions  "{DECISIONS}" \
    --out-dir    "{CORPUS_DEDUP}"

import pandas as pd
assert (CORPUS_DEDUP / 'entries.csv').exists(), 'entries not regenerated'
ent = pd.read_csv(CORPUS_DEDUP / 'entries.csv')
man = pd.read_csv(CORPUS_DEDUP / 'dedup_manifest.csv')
assert ent.entry_uid.notna().all() and ent.entry_uid.is_unique, 'entry_uid must be unique'
print(len(man), 'pages dropped;', len(ent), 'entries in the deduped corpus')
print(ent.groupby('volume').size())

In [ ]:
# Multimodal catalogue beside the deduped entries so --input-dir finds it.
# The pipeline prefers multimodal_clean.md (duplicate pages + degenerate crops removed,
# see multimodal_catalogue_<date>/clean_rules.py) over the raw multimodal.md.
import shutil
clean = CORPUS_DEDUP / 'multimodal_clean.md'
src = CORPUS_DIR / 'multimodal' / 'multimodal.md'
if clean.exists():
    print('cleaned catalogue present:', clean, '(used by the pipeline)')
elif src.exists():
    shutil.copy(src, CORPUS_DEDUP / 'multimodal.md')
    print('raw multimodal.md copied into', CORPUS_DEDUP, '— consider running the cleaning first')
else:
    print('no multimodal catalogue found — DwC-A will simply have no multimedia rows')

## Stage C · Knowledge graph for all 34 volumes (Gemini)

`configs/full_llm.yaml` has `sample.volume: null` (= all volumes), QA on, linking on.
With ~9.5k entries this is the long stage — the extraction alone is ~2–3 h at
concurrency 8 (every entry is a live call: the prompt changed, so the old cache
does not apply), then linking (GBIF / Wikidata / folk-name proposer, ~1–2 h on
the first pass), then graph build + SHACL (~15–30 min). All caches live on Drive,
so an interrupted run resumes where it stopped: just re-run cells 0, 1, C1, C3, C4.

**C2 (smoke test)** runs the first 25 entries live and prints what the model
extracted — eyeball it before you commit to the full run.

In [ ]:
# Pull the newest pipeline (restore the locally rewritten config first) and check the stage is there.
!cd /content/laubmann-kg_TP && git checkout -- configs/full_llm.yaml && git pull --ff-only && git log --oneline -1
import os
assert os.path.isdir('/content/laubmann-kg_TP/src/laubmann_kg/linking'), 'linking stage missing — GitHub main not updated'

In [ ]:
# C1 · Point every cache / review path at Drive (re-run after every git pull).
import yaml
for d in (LLM_CACHE, LINK_CACHE, EXPORTS_DIR / 'review'):
    d.mkdir(parents=True, exist_ok=True)

cfg = yaml.safe_load(open('configs/full_llm.yaml'))
cfg['extraction']['cache_dir'] = str(LLM_CACHE)

lk = cfg.setdefault('linking', {})
lk['enabled']    = True
lk['cache_dir']  = str(LINK_CACHE)                     # gbif_cache.json + wikidata_cache.json
lk['review_dir'] = str(EXPORTS_DIR / 'review')         # taxon_link_review.csv + person_link_review.csv (next to qa_flags.csv)
llm = lk.setdefault('taxa', {}).setdefault('llm', {})
llm['cache_dir'] = str(LINK_CACHE / 'llm')             # folk-name proposer cache
# After adjudicating a previous run's review CSVs, point these at the saved files and re-run C3/C4:
# lk['taxa']['reviewed_csv']    = str(EXPORTS_DIR / 'review' / 'taxon_link_review.csv')
# lk['persons']['reviewed_csv'] = str(EXPORTS_DIR / 'review' / 'person_link_review.csv')

yaml.safe_dump(cfg, open('configs/full_llm.yaml', 'w'), sort_keys=False, allow_unicode=True)
print('LLM cache   →', cfg['extraction']['cache_dir'])
print('link cache  →', lk['cache_dir'], '| proposer →', llm['cache_dir'])
print('review CSVs →', lk['review_dir'])
print('model:', cfg['extraction']['model'], '| thinking:', cfg['extraction'].get('thinking_level'),
      '| concurrency:', cfg['extraction'].get('concurrency'), '| max_output_tokens:', cfg['extraction'].get('max_output_tokens'))

In [ ]:
# C2 · SMOKE TEST — first 25 entries LIVE (about 1 minute). Read the table before starting the full run.
import yaml, json, pandas as pd
smoke_cfg = yaml.safe_load(open('configs/full_llm.yaml'))
smoke_cfg['sample']['limit'] = 25            # first N entries of the corpus (volume 1)
smoke_cfg['linking']['enabled'] = False      # extraction only
smoke_cfg['qa']['exclude'] = False           # show everything, flag only
yaml.safe_dump(smoke_cfg, open('configs/smoke_llm.yaml', 'w'), sort_keys=False, allow_unicode=True)

!laubmann-kg extract-observations --config configs/smoke_llm.yaml \
    --input-dir "{CORPUS_DEDUP}" --output-dir "{SMOKE_DIR}" 2>&1 | grep -vE " -> [0-9]+ observations$"

recs = json.load(open(SMOKE_DIR / 'observations.json'))
df = pd.DataFrame(recs)
print(len(df), 'observations from the first 25 entries\n')
cols = ['entry_id', 'entry_place', 'place', 'vernacular_de', 'scientific_name', 'taxon_rank',
        'individual_count', 'occurrence_status', 'sex', 'life_stage', 'breeding_evidence',
        'evidence', 'record_type', 'observer', 'confidence']
pd.set_option('display.max_colwidth', 28); pd.set_option('display.width', 250)
display(df[[c for c in cols if c in df.columns]].head(40))
print('\nentry places:', df.groupby('entry_id').entry_place.first().to_dict())

In [ ]:
# C3 · Full run: extraction (live, cached on Drive) → QA → linking → RDF/JSON-LD → SHACL.
# The per-entry progress lines are filtered out (Colab keeps only the last 5000 lines anyway);
# warnings, QA, linking and SHACL summaries stay visible.
!laubmann-kg export-jsonld --config configs/full_llm.yaml \
    --input-dir "{CORPUS_DEDUP}" --output-dir "{EXPORTS_DIR}" 2>&1 | grep --line-buffered -vE " -> [0-9]+ observations$"

In [ ]:
# C4 · Darwin Core Archive (replays extraction + linking from the caches: minutes, no LLM calls).
!laubmann-kg export-dwca --config configs/full_llm.yaml \
    --input-dir "{CORPUS_DEDUP}" --output-dir "{EXPORTS_DIR}" 2>&1 | grep --line-buffered -vE " -> [0-9]+ observations$"

In [ ]:
# C5 · Review deliverables: link adjudication CSVs + QA flags (all on Drive under EXPORTS_DIR/review).
import pandas as pd
from google.colab import files
rev = EXPORTS_DIR / 'review'
for name, col in [('taxon_link_review.csv', 'status'), ('person_link_review.csv', 'rule'), ('qa_flags.csv', 'reason')]:
    p = rev / name
    if p.exists():
        df = pd.read_csv(p)
        print(f'{name}: {len(df)} rows'); print(df[col].value_counts().to_string(), '\n')
    else:
        print(name, 'not found')
# Adjudicate: write y / yes / merge / 1 into the `decision` column of the correct candidate row,
# save back to the SAME Drive path, then enable the reviewed_csv lines in C1 and re-run C3 + C4.
for name in ['taxon_link_review.csv', 'person_link_review.csv']:
    if (rev / name).exists():
        files.download(str(rev / name))

## 3 · Check the graph — competency questions + metadata coverage

In [ ]:
import sys
sys.path.insert(0, 'src')
from laubmann_kg.kg.sparql import load_graph, run_all
from rdflib import RDF
from laubmann_kg.kg.rdf import LKG
import pandas as pd

g = load_graph(str(EXPORTS_DIR / 'rdf' / 'laubmann_sample.ttl'))
print('Triples:', len(g))
print('Entries:', len(set(g.subjects(RDF.type, LKG.DiaryEntry))))
print('Observations:', len(set(g.subjects(RDF.type, LKG.Observation))), '\n')   # ontology 0.4.0: lkg:Observation

for cq, rows in run_all(g).items():
    print(f'{cq}: {len(rows)} rows')

In [ ]:
# Metadata field coverage (ontology 0.4.0, DwC-first): how full is each property over all observations / entries?
from rdflib import RDF, Namespace
DWC = Namespace('http://rs.tdwg.org/dwc/terms/')
DWCIRI = Namespace('http://rs.tdwg.org/dwc/iri/')
obs = list(g.subjects(RDF.type, LKG.Observation))
ent = list(g.subjects(RDF.type, LKG.DiaryEntry))
def frac(nodes, pred):
    return sum(1 for o in nodes if g.value(o, pred) is not None) / max(len(nodes), 1)
print('observations:', len(obs), '| entries:', len(ent), '\n')
for name, pred in [('observedTaxon', LKG.observedTaxon), ('observedAt', LKG.observedAt),
                   ('hasLocality (own place)', LKG.hasLocality), ('dwc:individualCount', DWC.individualCount),
                   ('countQualifier', LKG.countQualifier), ('evidenceKind', LKG.evidenceKind),
                   ('hasVocalisation', LKG.hasVocalisation), ('dwc:behavior', DWC.behavior),
                   ('dwciri:habitat', DWCIRI.habitat), ('recordType', LKG.recordType),
                   ('dwciri:recordedBy', DWCIRI.recordedBy),
                   ('dwc:sex', DWC.sex), ('dwc:lifeStage', DWC.lifeStage),
                   ('breedingEvidence', LKG.breedingEvidence), ('movementKind', LKG.movementKind),
                   ('dwc:occurrenceStatus', DWC.occurrenceStatus)]:
    print(f'{name:26s} {frac(obs, pred):6.1%}')
print()
for name, pred in [('entryPlace', LKG.entryPlace), ('entryKind', LKG.entryKind),
                   ('hasWeather', LKG.hasWeather), ('containsTravelEvent', LKG.containsTravelEvent),
                   ('mentionsPerson', LKG.mentionsPerson), ('dwc:fieldNotes', DWC.fieldNotes)]:
    print(f'{name:26s} {frac(ent, pred):6.1%}')
tax = list(g.subjects(RDF.type, LKG.Taxon))
print()
for name, pred in [('dwc:scientificName', DWC.scientificName), ('dwc:family (GBIF)', DWC.family)]:
    print(f'{name:26s} {frac(tax, pred):6.1%}')

## 4 · Bundle the exports

In [ ]:
# Everything is already on Drive under EXPORTS_DIR; zip if you want a download.
!cd "{EXPORTS_DIR}/.." && zip -qr "{EXPORTS_DIR.name}.zip" "{EXPORTS_DIR.name}"
print('zip at', EXPORTS_DIR.parent / f'{EXPORTS_DIR.name}.zip')